In [27]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx

In [28]:
chla_oan_gems = gpd.read_file("../data/chla_oan_gems_dedup.geojson")

print(chla_oan_gems.shape)
chla_oan_gems.head()

(4651, 22)


,fecha,index_right,estacion,Decision,param,value,unit,depth,granularidad,fuente,...,id_estacion,nro_muestra,departamento,nombre_clave,uni_nombre,valor_original,limite_deteccion,limite_cuantificacion,valor_transformado,geometry
0,2018-11-12 23:20:00,0,URY00006,si,Chl-a,0.0000,mg/l,0.3,DIA,GEMS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (649386.009 6180413.463)
1,2018-11-12 08:30:00,1,URY00008,dudoso,Chl-a,0.0000,mg/l,0.3,DIA,GEMS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (664852.016 6189770.245)
2,2017-02-22 11:00:00,6,URY00029,si,Chl-a,0.0004,mg/l,0.3,DIA,GEMS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (570470.059 6220119.303)
3,2017-04-19 13:00:00,7,URY00029,si,Chl-a,0.0007,mg/l,0.3,DIA,GEMS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (570470.059 6220119.303)
4,2017-06-21 11:05:00,1694,XSLH030.S,NaN,NaN,NaN,NaN,NaN,NaN,OAN,...,100201.0,26124.0,FLORIDA,CloA_(lab),µg/L,<LD,0.600,1.5,<LD,POINT (570470.059 6220119.303)


In [29]:
chla_oan_gems.columns

Index(['fecha', 'index_right', 'estacion', 'Decision', 'param', 'value',
       'unit', 'depth', 'granularidad', 'fuente', 'dist', 'nombre_programa',
       'id_estacion', 'nro_muestra', 'departamento', 'nombre_clave',
       'uni_nombre', 'valor_original', 'limite_deteccion',
       'limite_cuantificacion', 'valor_transformado', 'geometry'],
      dtype='str')

In [30]:
oan_registros_clean = gpd.read_file("../data/oan/oan_registros_clean.geojson")

print(oan_registros_clean.shape)
oan_registros_clean.head()

(5019, 13)


,nombre_programa,estacion,id_estacion,nro_muestra,departamento,fecha_hora,nombre_clave,uni_nombre,valor_original,limite_deteccion,limite_cuantificacion,valor_transformado,geometry
0,Agua Arroyo Grande del Norte DCA,XGRN100.S,100626,29873,RÍO NEGRO,2019-09-05 16:00:00,CloA_(lab),µg/L,LD<x<LC,0.700000000,2.2,LD<x<LC,POINT (-57.41304 -32.9058)
1,Agua Arroyo Grande del Norte DCA,XGRN100.S,100626,29985,RÍO NEGRO,2019-10-31 09:40:00,CloA_(lab),µg/L,LD<x<LC,0.700000000,2.2,LD<x<LC,POINT (-57.41304 -32.9058)
2,Agua Arroyo Grande del Norte DCA,XGRN100.S,100626,31062,RÍO NEGRO,2020-06-04 14:20:00,CloA_(lab),µg/L,LD<X<LC,0.700000000,2.2,LD<X<LC,POINT (-57.41304 -32.9058)
3,Agua Arroyo Grande del Norte DCA,XGRN100.S,100626,31410,RÍO NEGRO,2020-08-06 12:46:00,CloA_(lab),µg/L,33.000000000,0.700000000,2.2,33.000000000,POINT (-57.41304 -32.9058)
4,Agua Arroyo Grande del Norte DCA,XGRN100.S,100626,31851,RÍO NEGRO,2020-11-19 12:29:00,CloA_(lab),µg/L,2.600000000,0.700000000,2.2,2.600000000,POINT (-57.41304 -32.9058)


In [31]:
gems_chla = gpd.read_file("../data/joins/gems_chla.geojson")

print(gems_chla.shape)
gems_chla.head()

(3796, 9)


,estacion,Decision,param,fecha,value,unit,depth,granularidad,geometry
0,URY00006,si,Chl-a,2018-11-12 23:20:00,0.0000,mg/l,0.3,DIA,POINT (-55.3727 -34.5071)
1,URY00008,dudoso,Chl-a,2018-11-12 08:30:00,0.0000,mg/l,0.3,DIA,POINT (-55.2061 -34.4204)
2,URY00029,si,Chl-a,2015-11-18 11:53:00,0.0030,mg/l,0.3,DIA,POINT (-56.2355 -34.15748)
3,URY00029,si,Chl-a,2016-01-20 11:20:00,0.0036,mg/l,0.3,DIA,POINT (-56.2355 -34.15748)
4,URY00029,si,Chl-a,2016-10-12 11:10:00,0.0015,mg/l,0.3,DIA,POINT (-56.2355 -34.15748)


### Cuantas mediciones de OAN y GEMS quedaron afuera del dedup?

`chla_oan_gems_dedup.geojson` solo tiene las mediciones de las estaciones que estan compartidas entre OAN y GEMS (ver oan_gems_normalization_v2.ipynb). Las mediciones de estaciones exclusivas de una sola fuente no entraron ahi. Vamos a contarlas.

In [32]:
oan_registros_clean["fuente"] = "OAN"
gems_chla["fuente"] = "GEMS"

oan_registros_clean = oan_registros_clean.rename(columns={"fecha_hora": "fecha"})
gems_chla = gems_chla[pd.to_datetime(gems_chla["fecha"]).dt.year >= 2017].copy()

oan_utm = oan_registros_clean.to_crs(32721).copy()
gems_utm = gems_chla.to_crs(32721).copy()

In [33]:
oan_stations = gpd.GeoDataFrame(geometry=oan_utm["geometry"].unique(), crs=oan_utm.crs)
gems_stations = gpd.GeoDataFrame(geometry=gems_utm["geometry"].unique(), crs=gems_utm.crs)

print(oan_stations.shape, gems_stations.shape)

(193, 1) (153, 1)


In [34]:
oan_match = gpd.sjoin_nearest(oan_utm, gems_stations, how="left", max_distance=0.1, distance_col="dist")
gems_match = gpd.sjoin_nearest(gems_utm, oan_stations, how="left", max_distance=0.1, distance_col="dist")

print(oan_match.shape, gems_match.shape)

(5019, 16) (3086, 12)


In [35]:
oan_exclusivas = oan_match[oan_match["index_right"].isna()].copy()
gems_exclusivas = gems_match[gems_match["index_right"].isna()].copy()

print("Mediciones OAN en estaciones no compartidas con GEMS:", oan_exclusivas.shape[0])
print("Mediciones GEMS en estaciones no compartidas con OAN:", gems_exclusivas.shape[0])

Mediciones OAN en estaciones no compartidas con GEMS: 788
Mediciones GEMS en estaciones no compartidas con OAN: 1084


In [36]:
oan_exclusivas["fuente"].value_counts()
gems_exclusivas["fuente"].value_counts()

print("Total OAN:", oan_registros_clean.shape[0], "| en dedup:", (chla_oan_gems["fuente"] == "OAN").sum(), "| exclusivas (afuera):", oan_exclusivas.shape[0])
print("Total GEMS:", gems_chla.shape[0], "| en dedup:", (chla_oan_gems["fuente"] == "GEMS").sum(), "| exclusivas (afuera):", gems_exclusivas.shape[0])

Total OAN: 5019 | en dedup: 3443 | exclusivas (afuera): 788
Total GEMS: 3086 | en dedup: 1208 | exclusivas (afuera): 1084


### Unir todo en un solo geojson, ordenado por fecha

Cargamos `chla_oan_gems_dedup.geojson` (estaciones compartidas, ya deduplicado) y le sumamos las mediciones exclusivas de OAN y de GEMS que quedaron afuera. El dedup no tiene columna `fecha` (se perdio al guardarlo, quedo como indice), asi que esas filas van a aparecer sin fecha al ordenar.

In [37]:
union_completo = gpd.GeoDataFrame(
    pd.concat([chla_oan_gems, oan_exclusivas, gems_exclusivas], ignore_index=True),
    geometry="geometry",
    crs=oan_utm.crs
)
union_completo = union_completo.sort_values("fecha").reset_index(drop=True)

print(union_completo.shape)
union_completo["fuente"].value_counts()

(6523, 22)


fuente
OAN     4231
GEMS    2292
Name: count, dtype: int64

In [38]:
union_completo.head()

,fecha,index_right,estacion,Decision,param,value,unit,depth,granularidad,fuente,...,id_estacion,nro_muestra,departamento,nombre_clave,uni_nombre,valor_original,limite_deteccion,limite_cuantificacion,valor_transformado,geometry
0,2017-01-02,NaN,MDP3M,NaN,NaN,NaN,NaN,NaN,NaN,OAN,...,100493.0,0.0,COLONIA,CloA_(lab),µg/L,8.900,0.100,0.1,8.900,POINT (401164.986 6214585.025)
1,2017-01-02,NaN,URY00077,si,Chl-a,0.0068,mg/l,0.3,DIA,GEMS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (678380.002 6147956.963)
2,2017-01-02,696.0,URY00076,si,Chl-a,0.0032,mg/l,0.3,DIA,GEMS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (679045.962 6144090.994)
3,2017-01-02,NaN,MDP5S,NaN,NaN,NaN,NaN,NaN,NaN,OAN,...,100497.0,0.0,COLONIA,CloA_(lab),µg/L,4.400,0.100,0.1,4.400,POINT (402233.023 6210401.038)
4,2017-01-02,NaN,MDP4S,NaN,NaN,NaN,NaN,NaN,NaN,OAN,...,100495.0,0.0,COLONIA,CloA_(lab),µg/L,5.900,0.100,0.1,5.900,POINT (401555.004 6213022.975)


In [39]:
union_completo

,fecha,index_right,estacion,Decision,param,value,unit,depth,granularidad,fuente,...,id_estacion,nro_muestra,departamento,nombre_clave,uni_nombre,valor_original,limite_deteccion,limite_cuantificacion,valor_transformado,geometry
0,2017-01-02 00:00:00,NaN,MDP3M,NaN,NaN,NaN,NaN,NaN,NaN,OAN,...,100493.0,0.0,COLONIA,CloA_(lab),µg/L,8.900,0.100,0.1,8.900,POINT (401164.986 6214585.025)
1,2017-01-02 00:00:00,NaN,URY00077,si,Chl-a,0.0068,mg/l,0.3,DIA,GEMS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (678380.002 6147956.963)
2,2017-01-02 00:00:00,696.0,URY00076,si,Chl-a,0.0032,mg/l,0.3,DIA,GEMS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (679045.962 6144090.994)
3,2017-01-02 00:00:00,NaN,MDP5S,NaN,NaN,NaN,NaN,NaN,NaN,OAN,...,100497.0,0.0,COLONIA,CloA_(lab),µg/L,4.400,0.100,0.1,4.400,POINT (402233.023 6210401.038)
4,2017-01-02 00:00:00,NaN,MDP4S,NaN,NaN,NaN,NaN,NaN,NaN,OAN,...,100495.0,0.0,COLONIA,CloA_(lab),µg/L,5.900,0.100,0.1,5.900,POINT (401555.004 6213022.975)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6518,2026-05-06 15:28:00,3160.0,NE81A.S,NaN,NaN,NaN,NaN,NaN,NaN,OAN,...,100650.0,2674998.0,RÍO NEGRO,CloA_(lab),µg/L,3.000000000,NaN,0.1,3.000000000,POINT (540435.203 6366112.119)
6519,2026-05-06 15:50:00,3093.0,NE82A.S,NaN,NaN,NaN,NaN,NaN,NaN,OAN,...,100649.0,2674997.0,RÍO NEGRO,CloA_(lab),µg/L,1.500000000,NaN,0.1,1.500000000,POINT (540534.657 6366265.697)
6520,2026-05-06 16:05:00,3294.0,NE77A.S,NaN,NaN,NaN,NaN,NaN,NaN,OAN,...,100647.0,2674996.0,RÍO NEGRO,CloA_(lab),µg/L,1.500000000,NaN,0.1,1.500000000,POINT (541076.43 6365906.268)
6521,2026-05-06 16:25:00,3361.0,NE72A.S,NaN,NaN,NaN,NaN,NaN,NaN,OAN,...,100646.0,2674995.0,RÍO NEGRO,CloA_(lab),µg/L,3.000000000,NaN,0.1,3.000000000,POINT (541711.453 6366698.147)


In [40]:
union_completo.to_file("../data/chla_oan_gems_union.geojson", driver="GeoJSON")